In [1]:
!pip install pennylane
! pip install aiohttp fsspec h5py
!pip install tqdm 


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


# 1) Downloading Hamiltonian
Change MAX_TERMS = 10 if you want more/less
However, note that the first 10 terms are all you need for basic testing. Following terms have lower coefficients and contribute less helpful information. Check the graphs on the github.

Note that this code cell lets you download Hamiltonians for one molecule at a time. Some of them are much larger than others. It is recommended that you parallelize this if you wish to download multiple at the same time. For a list of which ones are smaller (for faster testing), check the GitHub read me.

In [ ]:
import pennylane as qml
from tqdm import tqdm 
coefficients = []
operators = []
hamiltonian_chunks = []

# Download the dataset and retrieve the Hamiltonian chunks
ds = qml.data.load('other', name='ala')

# fiy if you want to make future changes
print(f"Dataset type: {type(ds)}")
print(f"Dataset content: {ds}")

# If ds is a list, iterate through it to find the actual dataset object
if isinstance(ds, list):
    # Take the first item if it's a list
    dataset = ds[0] if len(ds) > 0 else None
    if dataset is not None:
        print(f"First item type: {type(dataset)}")
        # Check if this object has list_attributes method
        if hasattr(dataset, 'list_attributes'):
            for key in dataset.list_attributes():
                if "hamiltonian" in key:
                    hamiltonian_chunks.append(getattr(dataset, key))
        elif hasattr(dataset, '__dict__'):
            # If no list_attributes, check the object's attributes directly
            for key in dir(dataset):
                if "hamiltonian" in key and not key.startswith('_'):
                    hamiltonian_chunks.append(getattr(dataset, key))
else:
    # If ds is not a list, use the original approach: IF they update it for some reason
    for key in ds.list_attributes():
        if "hamiltonian" in key:
            hamiltonian_chunks.append(getattr(ds, key))

# if we have hamiltonian chunks, we can proceed
if hamiltonian_chunks:
    # Combine all Hamiltonian chunks into a single string
    full_hamiltonian = "".join(hamiltonian_chunks)
    print('successfully combined')
    
    # Helper function to convert a string representation into a PennyLane operator
    def string_to_operator(op_string):
        if "Identity" in op_string:
            return qml.Identity(0)  # Identity defaults to acting on qubit 0
        
        terms = op_string.split(" @ ")  # Separate tensor product terms
        ops = []
        
        for term in terms:
            try:
                op, wire = term.split("(")
                wire = int(wire.strip(")"))  # Extract the qubit index
                if op == "X":
                    ops.append(qml.PauliX(wire))
                elif op == "Y":
                    ops.append(qml.PauliY(wire))
                elif op == "Z":
                    ops.append(qml.PauliZ(wire))
            except ValueError:
                continue  # Skip malformed lines
        
        return qml.prod(*ops) if len(ops) > 1 else ops[0]  # Create composite operator if needed
    
    # Process each line of the combined Hamiltonian string with progress bar
    lines = full_hamiltonian.split("\n")
    valid_lines = [line.strip() for line in lines if line.strip() and "Coefficient" not in line and "Operators" not in line]
    
    # Limit to first 10,000 terms for faster processing
    MAX_TERMS = 100
    if len(valid_lines) > MAX_TERMS:
        print(f'Found {len(valid_lines)} terms, limiting to first {MAX_TERMS} for faster processing')
        valid_lines = valid_lines[:MAX_TERMS]
    else:
        print(f'Processing all {len(valid_lines)} Hamiltonian terms...')
    
    for line in tqdm(valid_lines, desc="Building Hamiltonian", unit="terms"):
        parts = line.split()
        
        try:
            coeff = float(parts[0])  # Extract the coefficient
            op_string = " ".join(parts[1:])  # Extract the operators
            coefficients.append(coeff)
            operators.append(string_to_operator(op_string))
        except ValueError:
            continue  # Gracefully handle conversion errors
    
    # Build the PennyLane Hamiltonian
    print('Building final Hamiltonian object...')
    hamiltonian = qml.Hamiltonian(coefficients, operators)
    print('Hamiltonian successfully built!')
    print(f'Final Hamiltonian has {len(coefficients)} terms (limited to first {MAX_TERMS})')
else:
    print("No hamiltonian chunks found. Please check the dataset structure.")

Dataset type: <class 'list'>
Dataset content: [<Dataset = attributes: ['abbreviation', 'name', ...]>]
First item type: <class 'pennylane.data.base.dataset.Dataset'>
successfully combined
Found 2725840 terms, limiting to first 100 for faster processing


Building Hamiltonian: 100%|██████████| 100/100 [00:00<00:00, 49021.79terms/s]

Building final Hamiltonian object...
✓ Hamiltonian successfully built!
Final Hamiltonian has 100 terms (limited to first 100)
